# 01 Кросс-таблица_Профиль_Базовый соцдем_Частота телепросмотра #

- Задача: Посмотреть частоту телепросмотра по базовым целевым группам (по соц-дему и размеру населенного пункта)
- База: Профиль 2024
- Отчет: кросс-таблица
- Статистики: Universe, Col%, Row%, Affinity
- Количество дробных знаков:	1	
- Тотал по всем переменным (в строках и столбцах)		
- Пересечение переменных: 		
1. в столбцах:	
- Частота просмотра ТВ 	freq_tv_prof21 
2. в строках:	
- Пол	sex
- Возраст (6 интервалов)	age_7int
- Образование	ed_short
- Занятость	occup3
- Материальное положение семьи	family_income_short
- Размер семьи	houshold_num
- Наличие детей до 18 лет в семье	child_1
- Размер населенного пункта	city_size


In [ ]:
import pandas as pd
from brandpulse_api.calc import BrandpulseCalc

# имя файла с данными
DB_FILE = 'brandpulse3.duckdb'

# создаем объект для расчетов
bp = BrandpulseCalc(db_file=DB_FILE)

Справочно исследуем свойства

In [ ]:
# поиск по описанию
# bp.find_property(text='Пол')

# поиск по имени
# bp.find_property(sys_name='SEX')

# поиск по комбинации имени и описания
# bp.find_property(text='Возраст', sys_name='SEX')

Справочно исследуем значения свойств

In [ ]:
# поиск по описанию
#bp.find_property_options(text='Покупали кофе')

# поиск по имени
#bp.find_property_options(sys_name='ORDERED_FOOD_TO')

# поиск по комбинации имени и описания
#bp.find_property_options(text='кофе', sys_name='COFFEE')

# поиск по имени свойства (символ ^ добавлен для поиска имен, начинающихся с этой строки)
#bp.find_property_options(property_sys_name='^FOOD_DELIVERY')

Задаем параметры расчета

In [ ]:
profile_id = 3900499002973208128# id профиля 2024

# список соцдем свойств слева
property_left = ['SEX', 'AGE_7INT', 'ED_SHORT', 'OCCUP3', 'FAMILY_INCOME_SHORT', 'FAMILY_NUM', 'CHILD_1', 'CITY_SIZE']

# свойство сверху
property_up = 'FREQ_TV_PROF21'

Получаем данные расчета universe

In [ ]:
df_result = bp.calc_crosstab_property_universe(project_id=profile_id, property_sys_names_left=property_left, property_sys_name_up=property_up, demo=None)
df_result

Считаем размер сэмпла

In [ ]:
# добавляем расчет сэмпла
sample = bp.calc_sample(project_ids=[profile_id], demo=None)
sample

Считаем размер universe по всему профилю

In [ ]:
# добавляем расчет юниверс 
universe = bp.calc_universe(project_ids=[profile_id], demo=None)
print(universe)

Добавляем к рассчитанному universe расчет долей (row% и col%) и аффинити

In [ ]:
# добавляем доли и аффинити
df_tmp = df_result.copy()

num_cols = df_tmp.columns[4:]
non_num_cols = df_tmp.columns[:4]

row_sums = df_tmp[num_cols].sum(axis=1)

df_tmp['total_universe'] = row_sums
group_sum = df_tmp.groupby(df_tmp.columns[0])[num_cols].transform('sum')
group_sum_total_universe = df_tmp.groupby(df_tmp.columns[0])['total_universe'].transform('sum')

# добавляем строку сводных итогов
total_row = {
    non_num_cols[0]: 'total',
    non_num_cols[1]: '',
    non_num_cols[2]: '',
    non_num_cols[3]: '',
    'total_universe': group_sum_total_universe[0],
    'total_universe_col%': 100    
}
col_index = 0
for col in num_cols:
    total_row[col+'_universe'] = group_sum.iloc[0, col_index]
    total_row[col+'_col%'] = 100
    total_row[col+'_row%'] = 100 * group_sum.iloc[0, col_index] / group_sum_total_universe[0]
    total_row[col+'_affinity'] = 100
    col_index+=1

# считаем доли и аффинити    
result_tmp = df_tmp[non_num_cols].copy()
result_tmp['total_universe'] = df_tmp['total_universe']
result_tmp['total_universe_col%'] = 100 *df_tmp['total_universe'] / group_sum_total_universe

for col in num_cols:
    result_tmp[col+'_universe'] = df_tmp[col]
    result_tmp[col+'_col%'] = 100 * df_tmp[col] / group_sum[col]
    result_tmp[col+'_row%'] = 100 * df_tmp[col] / row_sums
    result_tmp[col+'_affinity'] = 100 * result_tmp[col+'_col%'] / result_tmp['total_universe_col%']
    
total_row_df = pd.DataFrame([total_row], columns=result_tmp.columns)
result = pd.concat([total_row_df, result_tmp], ignore_index=True)

# округляем до нужной точности
result['total_universe_col%'] = result['total_universe_col%'].round(1)
for col in num_cols:
    result[col+'_col%'] = result[col+'_col%'].round(1)
    result[col+'_row%'] = result[col+'_row%'].round(1)
    result[col+'_affinity'] = result[col+'_affinity'].round(0).astype(int)

result

Сохраняем в excel с формитированием отчета

In [ ]:
# сохранение в эксель с форматированием
import os
from openpyxl.styles import Alignment

subfolder = "excel"  # название подпапки
os.makedirs(subfolder, exist_ok=True)  # создаёт подпапку, если не существует

filepath = os.path.join(subfolder, "01_crosstab_profile_soc_dem_freq_tv.xlsx")

with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
    result.to_excel(writer, sheet_name='Расчет', startrow=8, startcol=1, index=False)
    
    # настройка листа экселя
    worksheet = writer.sheets['Расчет']
    
    # Устанавливаем ширину столбцов (учитываем startcol=1)
    worksheet.column_dimensions['B'].width = 20  
    worksheet.column_dimensions['C'].width = 20  
    worksheet.column_dimensions['D'].width = 20  
    worksheet.column_dimensions['E'].width = 20 
    worksheet.column_dimensions['F'].width = 20  
    worksheet.column_dimensions['J'].width = 20 
    
    worksheet['A1'] = 'База данных: Brandpulse 2024'
    worksheet['A2'] = 'Размер генеральной совокупности (тыс.): '
    worksheet['A3'] = 'Целевая база: Население <2024 год> [Профиль]'
    worksheet['A4'] = f'Размер целевой базы (тыс.): {universe}'
    worksheet['A5'] = 'Целевая группа: Население'
    worksheet['A6'] = f'Размер целевой группы (тыс.): {universe}     Выборка: {sample}'
    worksheet['A7'] = 'Размер (%): 100.0%'
    #worksheet['A1'].font = worksheet['A1'].font.copy(size=10)
    
    for cell in worksheet[9]:
        cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
    
    worksheet['B9'] = 'СОЦДЕМ'
    worksheet['C9'] = ''
    worksheet['D9'] = ''
    worksheet['E9'] = ''
    
    worksheet.row_dimensions[9].height = 90